In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, GammaRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import r2_score

import pickle
import json
import matplotlib.pyplot as plt
%matplotlib inline

# Загрузка модели

In [2]:
def load_model(model_path='model.pkl'):
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    return model

loaded_model = load_model()
print(f"Модель загружена: {loaded_model}")

Модель загружена: KNeighborsRegressor(n_neighbors=4, p=1)


# Преобразование данных

In [3]:
with open('config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

In [52]:
def prepare_data(data, config):
    X = data
    car_names = X['CarName'].str.split(' ', expand=True).fillna('')
    X['car_company'] = car_names[0]
    X['car_model'] = np.sum(car_names.loc[:, 1:], axis=1)

    X.drop(['CarName'], axis=1, inplace=True)
    
    num_cols = config['num_cols']
    str_cols = config['str_cols']
    bin_cols = config['bin_cols']
    cat_cols = config['cat_cols']

    X['car_company'] = X['car_company'].replace(
        {
            'maxda': 'mazda',
            'porcshce': 'porsche',
            'Nissan': 'nissan',
            'vokswagen': 'volkswagen',
            'vw': 'volkswagen',
            'toyouta': 'toyota',
        }
    )

    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

    X_train_encoded = enc.fit_transform(X[bin_cols + cat_cols])
    X[bin_cols + cat_cols] = X_train_encoded

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X

In [53]:
data = pd.DataFrame({
"symboling": [0],
"CarName": ["ford focus"],
"fueltype": ["gas"],
"aspiration": ["std"],
"doornumber": ["four"],
"carbody": ["sedan"],
"drivewheel": ["fwd"],
"enginelocation": ["front"],
"wheelbase": [104.3],
"carlength": [178.5],
"carwidth": [69.5],
"carheight": [57.9],
"curbweight": [2850],
"enginetype": ["ohc"],
"cylindernumber": ["four"],
"enginesize": [122],
"fuelsystem": ["mpfi"],
"boreratio": [3.31],
"stroke": [3.54],
"compressionratio": [10.0],
"horsepower": [120],
"peakrpm": [6000],
"citympg": [26],
"highwaympg": [36]
})

In [54]:
#data_to_predict = prepare_data('data/carprice_gen.csv',config=config)
data_to_predict = prepare_data(data,config=config)

In [56]:
predict = loaded_model.predict(data_to_predict)

In [57]:
predict

array([10378.25])

# Тест API

In [66]:
import requests

def predict_model(data):
    url = 'http://127.0.0.1:5000/get_predict'

    # Отправка POST-запроса с данными в формате форм-данных
    response = requests.post(url, json=data)

    # Проверка статуса ответа
    if response.status_code == 200:
        return response.json()
    else:
        return {"error": f"Request failed with status code {response.status_code}"}

# Пример данных для предсказания
data = {
  "symboling": 0,
  "CarName": "ford focus",
  "fueltype": "gas",
  "aspiration": "std",
  "doornumber": "four",
  "carbody": "sedan",
  "drivewheel": "fwd",
  "enginelocation": "front",
  "wheelbase": 104.3,
  "carlength": 178.5,
  "carwidth": 69.5,
  "carheight": 57.9,
  "curbweight": 2850,
  "enginetype": "ohc",
  "cylindernumber": "four",
  "enginesize": 122,
  "fuelsystem": "mpfi",
  "boreratio": 3.31,
  "stroke": 3.54,
  "compressionratio": 10.0,
  "horsepower": 120,
  "peakrpm": 6000,
  "citympg": 26,
  "highwaympg": 36
}

# Получение предсказания
prediction = predict_model(data)
print(prediction)

{'prediction': [10378.25]}
